In [6]:
# ============================================================
# CYBERSHIELD AI — COMPLETE SCAM COMMUNICATION DATASET AUDIT
# ============================================================
# Purpose:
# Strictly evaluate whether the dataset is suitable for:
#   1. Scam vs Legitimate detection
#   2. Scam-type classification
#   3. Multi-channel scam communication analysis
#
# IMPORTANT:
# This cell ONLY AUDITS the dataset.
# DO NOT train any ML model yet.
# ============================================================

import pandas as pd
import numpy as np
import os
import re
import unicodedata
from collections import Counter

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 300)
pd.set_option("display.width", 200)

# ------------------------------------------------------------
# 1. LOAD DATASET
# ------------------------------------------------------------

DATA_PATH = r"C:\Users\Admin\Desktop\CyberShield AI\ai_models\url_phishing\data\raw\sentinel_dataset_multiclass.csv"
print("=" * 80)
print("CYBERSHIELD AI — DATASET AUDIT STARTED")
print("=" * 80)

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"\nDataset not found!\n"
        f"Expected file: {DATA_PATH}\n"
        f"Current folder: {os.getcwd()}\n"
    )

df = pd.read_csv(DATA_PATH)

print("\n[1] DATASET LOADED")
print("-" * 80)
print("File:", DATA_PATH)
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Memory usage:", round(df.memory_usage(deep=True).sum() / 1024**2, 2), "MB")

print("\nColumn names:")
for i, col in enumerate(df.columns, 1):
    print(f"{i}. {col}")

# ------------------------------------------------------------
# 2. DATA TYPES
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("[2] DATA TYPES")
print("=" * 80)

print(df.dtypes)

# ------------------------------------------------------------
# 3. MISSING VALUES
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("[3] MISSING VALUES")
print("=" * 80)

missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_table = pd.DataFrame({
    "Missing_Count": missing,
    "Missing_Percentage": missing_pct
}).sort_values("Missing_Count", ascending=False)

print(missing_table)

# ------------------------------------------------------------
# 4. DUPLICATE ROWS
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("[4] DUPLICATE ROW ANALYSIS")
print("=" * 80)

exact_duplicates = df.duplicated().sum()

print("Exact duplicate rows:", exact_duplicates)
print("Duplicate percentage:",
      round(exact_duplicates / len(df) * 100, 2), "%")

# ------------------------------------------------------------
# 5. FIND TEXT / MESSAGE COLUMN
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("[5] TEXT / MESSAGE COLUMN DETECTION")
print("=" * 80)

possible_text_columns = [
    col for col in df.columns
    if any(keyword in col.lower()
           for keyword in [
               "text", "message", "body",
               "content", "subject", "sms"
           ])
]

print("Possible text columns:")
for col in possible_text_columns:
    print(" -", col)

if len(possible_text_columns) == 0:
    raise ValueError(
        "Could not automatically identify the message/text column."
    )

# Prefer a column literally named text
if "text" in df.columns:
    TEXT_COL = "text"
elif "message" in df.columns:
    TEXT_COL = "message"
else:
    TEXT_COL = possible_text_columns[0]

print("\nSelected text column:", TEXT_COL)

# ------------------------------------------------------------
# 6. TEXT QUALITY
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("[6] TEXT QUALITY ANALYSIS")
print("=" * 80)

df["_audit_text"] = df[TEXT_COL].fillna("").astype(str)

df["_text_length"] = df["_audit_text"].str.len()
df["_word_count"] = df["_audit_text"].str.split().str.len()

print("Empty messages:", (df["_audit_text"].str.strip() == "").sum())

print("\nCharacter length:")
print(df["_text_length"].describe())

print("\nWord count:")
print(df["_word_count"].describe())

print("\nMessages with < 3 words:",
      (df["_word_count"] < 3).sum())

print("Messages with < 5 characters:",
      (df["_text_length"] < 5).sum())

print("Messages > 500 characters:",
      (df["_text_length"] > 500).sum())

print("Messages > 1000 characters:",
      (df["_text_length"] > 1000).sum())

# ------------------------------------------------------------
# 7. UNIQUE MESSAGE ANALYSIS
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("[7] UNIQUE MESSAGE ANALYSIS")
print("=" * 80)

unique_messages = df["_audit_text"].nunique()

print("Total messages:", len(df))
print("Unique messages:", unique_messages)
print("Repeated messages:", len(df) - unique_messages)

print("Unique percentage:",
      round(unique_messages / len(df) * 100, 2), "%")

# ------------------------------------------------------------
# 8. NORMALIZED DUPLICATE ANALYSIS
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("[8] NORMALIZED DUPLICATE ANALYSIS")
print("=" * 80)

def normalize_text(text):
    text = str(text).lower()
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\s]", "", text)
    return text.strip()

df["_normalized_text"] = df["_audit_text"].apply(normalize_text)

normalized_unique = df["_normalized_text"].nunique()
normalized_duplicates = len(df) - normalized_unique

print("Unique normalized messages:", normalized_unique)
print("Normalized duplicates:", normalized_duplicates)
print(
    "Normalized duplicate percentage:",
    round(normalized_duplicates / len(df) * 100, 2),
    "%"
)

# ------------------------------------------------------------
# 9. LABEL / TARGET DETECTION
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("[9] LABEL / TARGET ANALYSIS")
print("=" * 80)

possible_label_columns = [
    col for col in df.columns
    if any(keyword in col.lower()
           for keyword in [
               "label", "target", "class",
               "category", "type"
           ])
]

print("Possible label/category columns:")
for col in possible_label_columns:
    print(" -", col)

for col in possible_label_columns:
    print("\n----------------------------------------")
    print("COLUMN:", col)
    print("Unique values:", df[col].nunique())
    print(df[col].value_counts(dropna=False).head(50))

# ------------------------------------------------------------
# 10. CATEGORY DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("[10] CATEGORY DISTRIBUTION")
print("=" * 80)

if "category_name" in df.columns:

    category_counts = df["category_name"].value_counts(dropna=False)

    category_table = pd.DataFrame({
        "Count": category_counts,
        "Percentage": (
            category_counts / len(df) * 100
        ).round(2)
    })

    print(category_table)

    print("\nNumber of categories:",
          df["category_name"].nunique())

else:
    print("category_name column not found.")

# ------------------------------------------------------------
# 11. BINARY SCAM VS LEGITIMATE DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("[11] SCAM VS LEGITIMATE ANALYSIS")
print("=" * 80)

if "label" in df.columns:

    label_counts = df["label"].value_counts(dropna=False)

    print(label_counts)

    print("\nPercentages:")
    print(
        (label_counts / len(df) * 100).round(2)
    )

    print("\nUnique label values:")
    print(df["label"].unique())

else:
    print("No 'label' column found.")

# ------------------------------------------------------------
# 12. CATEGORY × LABEL CROSS-TABULATION
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("[12] CATEGORY × LABEL CROSS-TABULATION")
print("=" * 80)

if "category_name" in df.columns and "label" in df.columns:

    cross_tab = pd.crosstab(
        df["category_name"],
        df["label"],
        margins=True
    )

    print(cross_tab)

else:
    print("Required columns not available.")

# ------------------------------------------------------------
# 13. TEXT DUPLICATES ACROSS DIFFERENT LABELS
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("[13] LABEL CONFLICT CHECK")
print("=" * 80)

if "label" in df.columns:

    labels_per_message = (
        df.groupby("_normalized_text")["label"]
        .nunique()
    )

    conflicting_messages = (
        labels_per_message > 1
    ).sum()

    print(
        "Same normalized message assigned to multiple labels:",
        conflicting_messages
    )

    if conflicting_messages > 0:
        print("\nWARNING: Label conflicts exist.")

else:
    print("Label column not available.")

# ------------------------------------------------------------
# 14. CATEGORY CONFLICT CHECK
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("[14] CATEGORY CONFLICT CHECK")
print("=" * 80)

if "category_name" in df.columns:

    categories_per_message = (
        df.groupby("_normalized_text")["category_name"]
        .nunique()
    )

    conflicting_categories = (
        categories_per_message > 1
    ).sum()

    print(
        "Same normalized message assigned to multiple categories:",
        conflicting_categories
    )

else:
    print("category_name not available.")

# ------------------------------------------------------------
# 15. URL / LINK PRESENCE
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("[15] URL / LINK ANALYSIS")
print("=" * 80)

url_pattern = r"(https?://|www\.|bit\.ly|tinyurl|t\.me/|wa\.me/)"

df["_has_url"] = (
    df["_audit_text"]
    .str.lower()
    .str.contains(url_pattern, regex=True, na=False)
)

print("Messages containing URLs/links:",
      df["_has_url"].sum())

print(
    "Percentage containing URLs:",
    round(df["_has_url"].mean() * 100, 2),
    "%"
)

# ------------------------------------------------------------
# 16. PHONE NUMBER PRESENCE
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("[16] PHONE NUMBER PATTERN ANALYSIS")
print("=" * 80)

phone_pattern = r"(?<!\d)(?:\+?\d[\d\s\-]{8,}\d)(?!\d)"

df["_has_phone"] = (
    df["_audit_text"]
    .str.contains(phone_pattern, regex=True, na=False)
)

print("Messages containing possible phone numbers:",
      df["_has_phone"].sum())

# ------------------------------------------------------------
# 17. EMAIL PRESENCE
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("[17] EMAIL PATTERN ANALYSIS")
print("=" * 80)

email_pattern = r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"

df["_has_email"] = (
    df["_audit_text"]
    .str.contains(email_pattern, regex=True, na=False)
)

print("Messages containing email addresses:",
      df["_has_email"].sum())

# ------------------------------------------------------------
# 18. MONEY / PAYMENT LANGUAGE
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("[18] FINANCIAL / PAYMENT SIGNAL ANALYSIS")
print("=" * 80)

financial_keywords = [
    "upi", "otp", "bank", "account",
    "kyc", "payment", "refund",
    "cash", "money", "loan",
    "investment", "lottery",
    "prize", "reward", "transaction",
    "card", "credit", "debit",
    "wallet", "nagad", "bkash"
]

text_lower = df["_audit_text"].str.lower()

keyword_counts = {}

for keyword in financial_keywords:
    keyword_counts[keyword] = text_lower.str.contains(
        rf"\b{re.escape(keyword)}\b",
        regex=True,
        na=False
    ).sum()

keyword_table = pd.DataFrame.from_dict(
    keyword_counts,
    orient="index",
    columns=["Message_Count"]
).sort_values("Message_Count", ascending=False)

keyword_table["Percentage"] = (
    keyword_table["Message_Count"] / len(df) * 100
).round(2)

print(keyword_table)

# ------------------------------------------------------------
# 19. SCAM-TYPE KEYWORD COVERAGE
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("[19] REQUIRED SCAM CATEGORY COVERAGE")
print("=" * 80)

required_categories = {
    "Banking": [
        "bank", "banking", "account"
    ],
    "OTP": [
        "otp", "one time password", "verification code"
    ],
    "UPI / Payment": [
        "upi", "payment", "pay", "transaction"
    ],
    "Investment": [
        "investment", "invest", "trading", "profit"
    ],
    "Lottery / Prize": [
        "lottery", "prize", "winner", "reward"
    ],
    "Romance": [
        "romance", "love", "dating", "relationship"
    ],
    "Delivery / Courier": [
        "parcel", "courier", "delivery", "shipment"
    ],
    "KYC": [
        "kyc", "verification"
    ],
    "Impersonation": [
        "police", "government", "officer",
        "official", "authority"
    ],
    "Refund": [
        "refund", "reimbursement"
    ],
    "Tech Support": [
        "technical support", "tech support",
        "customer support", "support"
    ],
    "Credential Theft": [
        "password", "login", "username",
        "credential", "sign in"
    ],
    "Social Engineering": [
        "urgent", "immediately", "verify",
        "suspended", "blocked", "act now"
    ]
}

coverage_results = []

for category, keywords in required_categories.items():

    pattern = "|".join(
        re.escape(k) for k in keywords
    )

    count = text_lower.str.contains(
        pattern,
        regex=True,
        na=False
    ).sum()

    coverage_results.append({
        "Required_Category": category,
        "Messages_Containing_Related_Terms": count,
        "Percentage": round(count / len(df) * 100, 2)
    })

coverage_table = pd.DataFrame(coverage_results)

print(coverage_table.to_string(index=False))

# ------------------------------------------------------------
# 20. LANGUAGE / CHARACTER ANALYSIS
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("[20] LANGUAGE / CHARACTER ANALYSIS")
print("=" * 80)

def count_unicode_ranges(text):
    latin = 0
    bengali = 0
    devanagari = 0
    other = 0

    for ch in str(text):

        code = ord(ch)

        if (
            0x0041 <= code <= 0x007A
            or 0x00C0 <= code <= 0x024F
        ):
            latin += 1

        elif 0x0980 <= code <= 0x09FF:
            bengali += 1

        elif 0x0900 <= code <= 0x097F:
            devanagari += 1

        elif ch.isalpha():
            other += 1

    return latin, bengali, devanagari, other


language_counts = {
    "Latin_characters": 0,
    "Bengali_characters": 0,
    "Devanagari_characters": 0,
    "Other_alphabetic": 0
}

for text in df["_audit_text"]:
    values = count_unicode_ranges(text)

    language_counts["Latin_characters"] += values[0]
    language_counts["Bengali_characters"] += values[1]
    language_counts["Devanagari_characters"] += values[2]
    language_counts["Other_alphabetic"] += values[3]

print(language_counts)

# ------------------------------------------------------------
# 21. SOURCE / CHANNEL COLUMNS
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("[21] SOURCE / CHANNEL INFORMATION")
print("=" * 80)

possible_source_columns = [
    col for col in df.columns
    if any(keyword in col.lower()
           for keyword in [
               "source", "channel",
               "platform", "dataset",
               "origin"
           ])
]

print("Possible source/channel columns:")

if possible_source_columns:

    for col in possible_source_columns:
        print("\nCOLUMN:", col)
        print("Unique values:", df[col].nunique())
        print(df[col].value_counts(dropna=False).head(30))

else:
    print("No obvious source/channel column found.")

# ------------------------------------------------------------
# 22. EXAMPLE MESSAGES PER CATEGORY
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("[22] REAL MESSAGE SAMPLE INSPECTION")
print("=" * 80)

if "category_name" in df.columns:

    for category in df["category_name"].dropna().unique():

        print("\n" + "-" * 80)
        print("CATEGORY:", category)
        print("-" * 80)

        samples = (
            df[df["category_name"] == category]
            [TEXT_COL]
            .sample(
                min(5, len(df[df["category_name"] == category])),
                random_state=42
            )
        )

        for i, message in enumerate(samples, 1):
            print(f"\n{i}. {message}")

# ------------------------------------------------------------
# 23. SHORTEST / LONGEST MESSAGES
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("[23] EXTREME TEXT EXAMPLES")
print("=" * 80)

print("\n--- 5 SHORTEST MESSAGES ---")

for text in (
    df.sort_values("_text_length")
      [TEXT_COL]
      .head(5)
):
    print("-", text)

print("\n--- 5 LONGEST MESSAGES ---")

for text in (
    df.sort_values("_text_length", ascending=False)
      [TEXT_COL]
      .head(5)
):
    print("-", text)

# ------------------------------------------------------------
# 24. CATEGORY SAMPLE COUNTS
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("[24] SMALL CATEGORY WARNING")
print("=" * 80)

if "category_name" in df.columns:

    counts = df["category_name"].value_counts()

    for category, count in counts.items():

        if count < 100:
            status = "🔴 VERY SMALL"

        elif count < 300:
            status = "🟠 SMALL"

        elif count < 500:
            status = "🟡 MODERATE"

        else:
            status = "🟢 GOOD"

        print(
            f"{category:35s} {count:6d} samples   {status}"
        )

# ------------------------------------------------------------
# 25. BASIC QUALITY SCORE
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("[25] AUTOMATIC PRELIMINARY QUALITY CHECK")
print("=" * 80)

score = 0
max_score = 0
reasons = []

# Dataset size
max_score += 1

if len(df) >= 10000:
    score += 1
    reasons.append("🟢 Dataset has at least 10,000 records.")
elif len(df) >= 5000:
    score += 0.5
    reasons.append("🟡 Dataset has 5,000–9,999 records.")
else:
    reasons.append("🔴 Dataset has fewer than 5,000 records.")

# Text availability
max_score += 1

if (df["_audit_text"].str.strip() != "").mean() >= 0.95:
    score += 1
    reasons.append("🟢 Text coverage is strong.")
else:
    reasons.append("🔴 Too many empty/invalid text records.")

# Unique messages
max_score += 1

unique_ratio = df["_normalized_text"].nunique() / len(df)

if unique_ratio >= 0.90:
    score += 1
    reasons.append("🟢 High message uniqueness.")
elif unique_ratio >= 0.75:
    score += 0.5
    reasons.append("🟡 Moderate message uniqueness.")
else:
    reasons.append("🔴 Large duplicate/repetition problem.")

# Label
max_score += 1

if "label" in df.columns and df["label"].nunique() >= 2:
    score += 1
    reasons.append("🟢 Multiple target classes exist.")
else:
    reasons.append("🔴 Target labels are missing or invalid.")

# Categories
max_score += 1

if "category_name" in df.columns and df["category_name"].nunique() >= 3:
    score += 1
    reasons.append("🟢 Multiple scam categories exist.")
else:
    reasons.append("🔴 Insufficient category structure.")

# ------------------------------------------------------------
# FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("[26] PRELIMINARY AUDIT SCORE")
print("=" * 80)

print(
    f"Score: {score}/{max_score}"
)

print(
    f"Percentage: {round(score/max_score*100, 2)}%"
)

print("\nReasons:")

for reason in reasons:
    print(reason)

print("\n" + "=" * 80)
print("IMPORTANT: THIS IS NOT THE FINAL DATASET VERDICT")
print("=" * 80)

print("""
The automatic score is only a preliminary check.

FINAL HUMAN/RESEARCH REVIEW MUST CONSIDER:

1. Are the messages genuinely realistic?
2. Are the labels trustworthy?
3. Are categories actually distinct?
4. Are important scam categories missing?
5. Is the dataset heavily templated?
6. Is there synthetic/AI-generated content?
7. Is there train/test leakage risk?
8. Are legitimate examples sufficiently diverse?
9. Are WhatsApp/Telegram/SMS/email represented?
10. Does the dataset match the actual CyberShield objective?

Send the COMPLETE OUTPUT of this cell for the final decision.
""")

# ------------------------------------------------------------
# CLEANUP AUDIT-ONLY COLUMNS
# ------------------------------------------------------------

audit_columns = [
    "_audit_text",
    "_text_length",
    "_word_count",
    "_normalized_text",
    "_has_url",
    "_has_phone",
    "_has_email"
]

# Keep original df untouched for later work
audit_df = df.copy()

print("\nAudit completed successfully.")
print("Original dataset shape:", df.shape)

CYBERSHIELD AI — DATASET AUDIT STARTED

[1] DATASET LOADED
--------------------------------------------------------------------------------
File: C:\Users\Admin\Desktop\CyberShield AI\ai_models\url_phishing\data\raw\sentinel_dataset_multiclass.csv
Rows: 14000
Columns: 5
Memory usage: 4.26 MB

Column names:
1. id
2. text
3. label
4. category
5. category_name

[2] DATA TYPES
id               int64
text               str
label            int64
category         int64
category_name      str
dtype: object

[3] MISSING VALUES
               Missing_Count  Missing_Percentage
id                         0                 0.0
text                       0                 0.0
label                      0                 0.0
category                   0                 0.0
category_name              0                 0.0

[4] DUPLICATE ROW ANALYSIS
Exact duplicate rows: 0
Duplicate percentage: 0.0 %

[5] TEXT / MESSAGE COLUMN DETECTION
Possible text columns:
 - text

Selected text column: text

[6] 

C:\Users\Admin\AppData\Local\Temp\ipykernel_8516\817084364.py:381: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(url_pattern, regex=True, na=False)


Messages containing possible phone numbers: 909

[17] EMAIL PATTERN ANALYSIS
Messages containing email addresses: 0

[18] FINANCIAL / PAYMENT SIGNAL ANALYSIS
             Message_Count  Percentage
bank                  3990       28.50
account               3219       22.99
card                  1699       12.14
otp                   1245        8.89
transaction           1198        8.56
upi                   1050        7.50
payment                945        6.75
kyc                    761        5.44
refund                 507        3.62
money                  484        3.46
credit                 479        3.42
wallet                 372        2.66
reward                 312        2.23
debit                  272        1.94
cash                   179        1.28
loan                   166        1.19
prize                  160        1.14
investment             156        1.11
lottery                 77        0.55
nagad                    0        0.00
bkash                  